[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/04-object-detection.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/04-object-detection.ipynb)

# Module 6.4 — Object Detection
**Module 6: Computer Vision** | Estimated time: 45 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Apply Haar cascades for face detection using `cv2.CascadeClassifier`
- Understand and use the HOG descriptor for pedestrian detection
- Implement IoU calculation and Non-Maximum Suppression (NMS)
- Explain YOLO's architecture and the differences between versions
- Run YOLOv8 nano inference on a sample image with the `ultralytics` library

In [ ]:
!pip install opencv-python-headless --quiet
!pip install ultralytics --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests, os, time

print(f'OpenCV {cv2.__version__}')

os.makedirs('/tmp/cv_detect', exist_ok=True)

def show_bgr(img, title='', figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off')
    plt.tight_layout(); plt.show()

# Download a sample face image (Lena — classic CV test image)
face_url = 'https://upload.wikimedia.org/wikipedia/en/7/7d/Lena.png'
r = requests.get(face_url)
with open('/tmp/cv_detect/face.png', 'wb') as f:
    f.write(r.content)

# Download a sample street scene for YOLO
street_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/320px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg'
r2 = requests.get(street_url)
with open('/tmp/cv_detect/scene.jpg', 'wb') as f:
    f.write(r2.content)
print('Images downloaded.')

## Haar Cascades for Face Detection

Haar cascades were introduced by Viola and Jones in 2001. They use a sliding window approach with a cascade of weak classifiers trained on Haar-like features (rectangular patterns of light and dark regions).

**Strengths:** Very fast, real-time performance, no GPU needed.  
**Weaknesses:** Sensitive to pose, lighting, and occlusion. Tends to produce false positives.

OpenCV ships with pre-trained cascade XML files for faces, eyes, smiles, and more.

In [ ]:
# Download the frontal face cascade XML from OpenCV's GitHub
cascade_url = ('https://raw.githubusercontent.com/opencv/opencv/master/'
               'data/haarcascades/haarcascade_frontalface_default.xml')
r = requests.get(cascade_url)
with open('/tmp/cv_detect/haarcascade_frontalface_default.xml', 'wb') as f:
    f.write(r.content)

# Also download eye cascade
eye_url = ('https://raw.githubusercontent.com/opencv/opencv/master/'
           'data/haarcascades/haarcascade_eye.xml')
r2 = requests.get(eye_url)
with open('/tmp/cv_detect/haarcascade_eye.xml', 'wb') as f:
    f.write(r2.content)

print('Cascades downloaded.')

# Load image and cascades
face_img = cv2.imread('/tmp/cv_detect/face.png')
if face_img is None:
    face_img = np.ones((300, 250, 3), dtype=np.uint8) * 180
    cv2.ellipse(face_img, (125, 130), (80, 100), 0, 0, 360, (200, 170, 140), -1)
    cv2.circle(face_img, (100, 110), 12, (50, 50, 50), -1)
    cv2.circle(face_img, (150, 110), 12, (50, 50, 50), -1)
    print('Using synthetic face.')

face_cascade = cv2.CascadeClassifier(
    '/tmp/cv_detect/haarcascade_frontalface_default.xml')
eye_cascade  = cv2.CascadeClassifier(
    '/tmp/cv_detect/haarcascade_eye.xml')

gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)

# Detect faces
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,    # pyramid scale: how much to reduce image each step
    minNeighbors=5,     # how many neighbours a candidate must have
    minSize=(30, 30)
)
print(f'Faces detected: {len(faces)}')

result = face_img.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 3)
    cv2.putText(result, 'Face', (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    # Detect eyes within face ROI
    roi_gray = gray[y:y+h, x:x+w]
    roi_color = result[y:y+h, x:x+w]
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1,
                                        minNeighbors=10)
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

show_bgr(result, f'Haar Cascade — {len(faces)} face(s) detected')

## HOG Descriptor and Pedestrian Detection

**HOG** (Histogram of Oriented Gradients) describes the distribution of gradient orientations in local regions of an image. It captures shape and texture information robustly.

OpenCV provides a built-in HOG + SVM pedestrian detector trained on the INRIA Person Dataset. It detects upright people in images.

In [ ]:
# HOG descriptor parameters
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Visualise what HOG features look like on our scene image
scene = cv2.imread('/tmp/cv_detect/scene.jpg')
if scene is None:
    scene = np.random.randint(80, 180, (400, 640, 3), dtype=np.uint8)
    # Add some rectangular shapes (simulate people)
    for x in [100, 300, 500]:
        cv2.rectangle(scene, (x, 80), (x+60, 260), (120, 90, 60), -1)

scene_gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)

# Run HOG pedestrian detection
rects, weights = hog.detectMultiScale(
    scene,
    winStride=(8, 8),
    padding=(4, 4),
    scale=1.05
)
print(f'HOG detected {len(rects)} pedestrian candidate(s)')

hog_result = scene.copy()
for i, (x, y, w, h) in enumerate(rects):
    cv2.rectangle(hog_result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(hog_result, f'{weights[i][0]:.2f}', (x, y-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

show_bgr(hog_result, f'HOG Pedestrian Detector — {len(rects)} detections')

# Compute a HOG feature vector for one patch
patch = cv2.resize(scene_gray[:128, :64], (64, 128))
hog_single = cv2.HOGDescriptor((64,128),(16,16),(8,8),(8,8),9)
descriptor = hog_single.compute(patch)
print(f'HOG descriptor length: {len(descriptor)} (64×128 window)')

## Non-Maximum Suppression (NMS)

Object detectors often produce multiple overlapping bounding boxes for a single object. **NMS** removes redundant boxes by:
1. Sorting boxes by confidence score (highest first)
2. Keeping the highest-confidence box
3. Removing all boxes that overlap the kept box above an IoU threshold
4. Repeating until no boxes remain

**IoU** (Intersection over Union) measures overlap between two boxes:
```
IoU = Area(Intersection) / Area(Union)
```

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0

def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """Apply NMS. boxes: list of [x1,y1,x2,y2], scores: list of floats."""
    if not boxes:
        return []
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    kept = []
    while order:
        i = order.pop(0)
        kept.append(i)
        order = [j for j in order
                 if compute_iou(boxes[i], boxes[j]) < iou_threshold]
    return kept

# Demonstrate NMS with synthetic overlapping boxes
np.random.seed(42)
boxes_raw = [
    [50, 50, 200, 200],
    [60, 60, 210, 210],
    [55, 55, 195, 195],
    [300, 100, 450, 280],
    [310, 110, 460, 290],
    [100, 300, 250, 420],
]
scores_raw = [0.95, 0.85, 0.80, 0.90, 0.75, 0.88]

kept_idx = non_max_suppression(boxes_raw, scores_raw, iou_threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, idx_list, title in zip(
        axes,
        [list(range(len(boxes_raw))), kept_idx],
        [f'Before NMS ({len(boxes_raw)} boxes)', f'After NMS ({len(kept_idx)} boxes)']):
    ax.set_xlim(0, 500); ax.set_ylim(0, 500)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(title)
    for i in idx_list:
        x1, y1, x2, y2 = boxes_raw[i]
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f'{scores_raw[i]:.2f}', color='blue', fontsize=9)
plt.tight_layout(); plt.show()

print(f'IoU between box0 and box1: {compute_iou(boxes_raw[0], boxes_raw[1]):.3f}')
print(f'Kept indices after NMS: {kept_idx}')

## YOLO Architecture Overview

YOLO (You Only Look Once) is a family of real-time object detectors that process the entire image in a single forward pass.

**Key ideas:**
- Divide image into an S×S grid
- Each grid cell predicts B bounding boxes and class probabilities
- Predictions made simultaneously → very fast (real-time capable)
- Anchor boxes encode common aspect ratios

**Version history:**
| Version | Year | Key improvement |
|---|---|---|
| YOLOv1 | 2016 | Original single-pass detection |
| YOLOv3 | 2018 | Multi-scale detection with FPN |
| YOLOv5 | 2020 | PyTorch rewrite, very popular |
| YOLOv8 | 2023 | Ultralytics rewrite, segmentation + pose |
| YOLOv11 | 2024 | Further efficiency improvements |

In [ ]:
# Explain YOLO grid concept visually
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, 7); ax.set_ylim(0, 7)
ax.set_aspect('equal')

# Draw grid
for i in range(8):
    ax.axhline(i, color='gray', linewidth=0.5)
    ax.axvline(i, color='gray', linewidth=0.5)

# Highlight a responsible grid cell
from matplotlib.patches import FancyArrow
rect = patches.Rectangle((2, 3), 1, 1, linewidth=3,
                          edgecolor='red', facecolor='#ffcccc', alpha=0.7)
ax.add_patch(rect)

# Draw a bounding box prediction
bbox = patches.Rectangle((1.2, 1.5), 3.5, 3.0, linewidth=2,
                          edgecolor='blue', facecolor='none', linestyle='--')
ax.add_patch(bbox)

ax.plot(2.5, 3.5, 'r*', markersize=15, label='Object centre (responsible cell)')
ax.set_title('YOLO Grid — each cell predicts bounding boxes\nfor objects whose centre falls in it',
             fontsize=12)
ax.legend(loc='upper right')
ax.set_xlabel('Grid columns'); ax.set_ylabel('Grid rows')
plt.tight_layout(); plt.show()

print('YOLO output tensor shape example (YOLOv3, 416×416 input):')
print('  Large objects:  13×13×(3×(5+C)) grid')
print('  Medium objects: 26×26×(3×(5+C)) grid')
print('  Small objects:  52×52×(3×(5+C)) grid')
print('  where C = number of classes, 5 = (x,y,w,h,confidence)')

## YOLOv8 Inference with Ultralytics

The `ultralytics` library provides a clean, high-level API for YOLOv8. The nano model (`yolov8n`) is the smallest variant — fast to download and run, perfect for learning.

In [ ]:
from ultralytics import YOLO
import PIL.Image

# Load YOLOv8 nano — weights downloaded automatically on first run (~6 MB)
model = YOLO('yolov8n.pt')
print(f'Model loaded: {model.info()[0]} parameters')

# Run inference on our scene image
results = model('/tmp/cv_detect/scene.jpg',
                conf=0.25,   # confidence threshold
                iou=0.45,    # NMS IoU threshold
                verbose=False)

# Visualise
result = results[0]
print(f'Detections: {len(result.boxes)}')
if len(result.boxes) > 0:
    for box in result.boxes:
        cls_id  = int(box.cls[0])
        conf    = float(box.conf[0])
        label   = model.names[cls_id]
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        print(f'  {label:20s} conf={conf:.2f}  box=[{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]')

# Plot using ultralytics built-in visualiser
plotted = result.plot()   # returns BGR numpy array
show_bgr(plotted, 'YOLOv8 Nano — Object Detection')

## Summary

| Method | Speed | Accuracy | Best for |
|---|---|---|---|
| Haar Cascade | Very fast | Low-medium | Face detection in controlled conditions |
| HOG + SVM | Fast | Medium | Pedestrian detection |
| YOLO | Fast–medium | High | General real-time detection |
| NMS | — | — | Removing duplicate detections from any detector |

## Practice Exercises

**Exercise 1 — Multi-scale Face Detection:**  
Download a group photo. Run `detectMultiScale` with three different `scaleFactor` values (1.05, 1.1, 1.3) and `minNeighbors` values (3, 5, 8). Plot the number of detections vs. false-positives for each combination.

**Exercise 2 — Custom NMS:**  
Create 20 random overlapping bounding boxes with random confidence scores (use `np.random.rand`). Run your `non_max_suppression` function with IoU thresholds of 0.3, 0.5, and 0.7. Plot how the number of kept boxes changes.

**Exercise 3 — YOLOv8 on Your Own Image:**  
Upload any image to Colab using `from google.colab import files; uploaded = files.upload()`. Run YOLOv8 on it, then manually verify the detections by listing each class, confidence, and bounding box coordinates in a formatted table.